In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import pymysql
from dotenv import load_dotenv
import os

# Cargar variables de entorno desde .env
load_dotenv()

# Configurar credenciales de la base de datos
DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'port': int(os.getenv('DB_PORT', 3306))
}

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Ruta a los datos clasificados
INTERIM_PATH = Path('../../data/interim')
print(f"Ruta de datos: {INTERIM_PATH.resolve()}")

Ruta de datos: C:\Users\David Merlez\Desktop\ConceptClassifier\ConceptClassifier\data\interim


In [2]:
def load_all_classified_data(interim_path):
    """Carga todos los archivos JSON clasificados y extrae la información relevante."""
    all_data = []
    files = list(interim_path.glob('classified_*.json'))
    
    print(f"Encontrados {len(files)} archivos clasificados")
    
    for file in files:
        try:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extraer user_id del nombre del archivo
            user_id = file.stem.replace('classified_', '').split('_')[1] if 'user_' in file.stem else 'unknown'
            
            for transaction in data:
                # Extraer datos de la cuenta bancaria
                account = transaction.get('account', '')
                bank_name = transaction.get('bank_name', '')
                bank_id = transaction.get('bank_id', '')
                
                for concept_item in transaction.get('classified_concepts', []):
                    concept = concept_item.get('concept', '')
                    classification = concept_item.get('classification', {})
                    sub_category = classification.get('sub_category', 'unknown')
                    main_category = classification.get('category', 'unknown')
                    
                    # Extraer monto de la transacción (convertir de céntimos a euros)
                    amount_cents = transaction.get('amount', 0)
                    amount_euros = amount_cents / 100.0 if amount_cents else 0
                    
                    # Extraer fecha de la transacción (value_date es la fecha oficial)
                    date = transaction.get('value_date', '')
                    
                    # Tipo de transacción (ingreso/gasto)
                    tx_type = 'Ingreso' if amount_euros > 0 else 'Gasto'
                    
                    all_data.append({
                        'user_id': user_id,
                        'account': account, 
                        'bank_name': bank_name,
                        'bank_id': bank_id,
                        'date': date,
                        'concept': concept,
                        'original_concept': concept_item.get('original', ''),
                        'main_category': main_category,
                        'sub_category': sub_category,
                        'amount_euros': amount_euros,
                        'amount_abs': abs(amount_euros),
                        'tx_type': tx_type,
                        'file': file.name
                    })
        except Exception as e:
            print(f"Error procesando {file.name}: {e}")
    
    return pd.DataFrame(all_data)

# Cargar datos
df = load_all_classified_data(INTERIM_PATH)
print(f"\nTotal de conceptos cargados: {len(df):,}")
print(f"Total de usuarios: {df['user_id'].nunique():,}")

Encontrados 20614 archivos clasificados

Total de conceptos cargados: 13,450,353
Total de usuarios: 20,002


In [ ]:
def get_loan_data_for_accounts(df):
    """Obtiene datos de préstamos solo para las cuentas que existen en el DataFrame"""
    try:
        # Obtener lista única de cuentas bancarias del DataFrame
        accounts_list = df['account'].dropna().unique().tolist()
        
        if not accounts_list:
            print("No hay cuentas bancarias en el DataFrame")
            return None
        
        print(f"Total de cuentas únicas a buscar: {len(accounts_list):,}")
        
        # Conectar a la base de datos
        connection = pymysql.connect(**DB_CONFIG)
        print("Conexión exitosa a la base de datos")
        
        # Crear placeholders para el IN clause
        placeholders = ','.join(['%s'] * len(accounts_list))
        
        # Query con filtro por cuentas
        query = f"""
        select 
            ba.full_account_number, 
            la.id as loan_application_id,
            la.created_at as loan_created_at,
            la.ref_id,
            u.dni,
            la.status,
            la.is_returning,
            la.amount,
            la.due_date,
            la.close_date,
            case
                when la.status = 'CLOSED' 
                    then datediff(la.close_date, la.due_date)
                when la.status = 'DELINQUENT' 
                    then datediff(current_date(), la.due_date)
                else null
            end as dr
        from available_bank_accounts ba 
        join users u on ba.user_id = u.id
        join loan_applications la on ba.user_id = la.user_id
        where la.status in ('CLOSED', 'DELINQUENT')
        and ba.full_account_number IN ({placeholders})
        order by ba.created_at desc
        """
        
        # Ejecutar query con parámetros
        df_loans = pd.read_sql(query, connection, params=accounts_list)
        
        print(f"\nTotal de registros obtenidos: {len(df_loans):,}")
        print(f"Cuentas con match: {df_loans['full_account_number'].nunique():,}")
        
        return df_loans
        
    except Exception as e:
        print(f"Error al conectar a la base de datos: {e}")
        return None
        
    finally:
        if 'connection' in locals():
            connection.close()
            print("Conexión cerrada")
# Obtener datos de SQL solo para cuentas que existen en df
df_loans = get_loan_data_for_accounts(df)

if df_loans is not None:
    print("\nPrimeras filas:")
    display(df_loans.head())
    
    print("\nColumnas:")
    print(df_loans.columns.tolist())

Total de cuentas únicas a buscar: 27,653
Conexión exitosa a la base de datos


C:\Users\David Merlez\AppData\Local\Temp\ipykernel_19652\249539625.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_loans = pd.read_sql(query, connection, params=accounts_list)



Total de registros obtenidos: 40,582
Cuentas con match: 12,148
Conexión cerrada

Primeras filas:


,full_account_number,loan_application_id,loan_created_at,ref_id,dni,status,is_returning,amount,due_date,close_date,dr
0,ES7621004525932200277228,10314317,2026-01-21 19:23:22,28809985461642555283,Y8083609C,CLOSED,0,50.0,2026-02-28,2026-02-28,0
1,ES2021004525912200260583,10314317,2026-01-21 19:23:22,28809985461642555283,Y8083609C,CLOSED,0,50.0,2026-02-28,2026-02-28,0
2,ES4215830001179046526763,10301791,2026-01-16 14:10:12,35721675887799545950,X5637815D,CLOSED,0,200.0,2026-02-15,2026-02-28,13
3,ES1801825322250207907920,10375134,2026-02-14 10:52:43,37607867378588791497,Z1199790T,CLOSED,0,150.0,2026-03-16,2026-02-18,-26
4,ES8621004972642200337405,10330621,2026-01-28 11:38:06,64728211828741045917,Z3388410D,CLOSED,0,200.0,2026-02-27,2026-02-26,-1



Columnas:
['full_account_number', 'loan_application_id', 'loan_created_at', 'ref_id', 'dni', 'status', 'is_returning', 'amount', 'due_date', 'close_date', 'dr']


In [4]:
df_loans.head(1)

,full_account_number,loan_application_id,loan_created_at,ref_id,dni,status,is_returning,amount,due_date,close_date,dr
0,ES7621004525932200277228,10314317,2026-01-21 19:23:22,28809985461642555283,Y8083609C,CLOSED,0,50.0,2026-02-28,2026-02-28,0


In [5]:
# Hacer merge entre los datos clasificados (df) y los datos de préstamos (df_loans)
df_merged = df.merge(
    df_loans, 
    left_on='account', 
    right_on='full_account_number', 
    how='inner'  # 'inner' solo mantiene registros con match en ambas tablas
)